In [0]:
# Vamos a configurar y llamar a la API.
from pyspark.sql.functions import col, lit
from datetime import datetime

# Verificar si ya ingestamos datos hoy
ya_ingestado = False
try:
    conteo = spark.table("nasa_dw.bronze.neo_raw") \
    .filter(col("ingestion_timestamp").startswith(datetime.today().strftime("%Y-%m-%d"))) \
    .count()
    ya_ingestado = conteo > 0
except:
    ya_ingestado = False

if ya_ingestado:
    print("⏭️ Ya se ingestó hoy, saltando Bronze...")
    dbutils.notebook.exit("Ya ingestado hoy")

import requests
from datetime import datetime, timedelta

NASA_API_KEY = "TU_API_ACÁ"  # ← reemplazá con tu key
BASE_URL     = "https://api.nasa.gov/neo/rest/v1/feed"

fecha_fin    = datetime.today().strftime("%Y-%m-%d")
fecha_inicio = "2024-01-01"

response = requests.get(BASE_URL, params={
    "start_date": fecha_inicio,
    "end_date":   fecha_fin,
    "api_key":    NASA_API_KEY
})
response.raise_for_status()
data = response.json()

# Validate API response structure
if "near_earth_objects" not in data:
    error_detail = data.get("error_message") or data.get("error") or str(data)
    raise ValueError(f"API response missing 'near_earth_objects'. API returned: {error_detail}")

print(f"✅ Asteroides encontrados: {data['element_count']}")
print(f"   Período: {fecha_inicio} → {fecha_fin}")

In [0]:
# "Aplanamos" el JSON, es decir, vamos acomodando los valores.
registros = []

# Validate API response structure
if "near_earth_objects" not in data:
    raise ValueError(
        "ERROR: La variable 'data' no contiene 'near_earth_objects'.\n\n"
        "Causas posibles:\n"
        "1. La celda anterior terminó antes de llamar a la API (revisa si hay dbutils.notebook.exit())\n"
        "2. Otra variable sobrescribió 'data'\n\n"
        "Solución: Ejecuta la celda anterior primero y asegúrate de que no hay 'exit()' activo."
    )

near_earth_objects = data["near_earth_objects"]
assert isinstance(near_earth_objects, dict), f"Expected dict, got {type(near_earth_objects)}"

for fecha, asteroides in near_earth_objects.items():
    for a in asteroides:
        acercamiento = a["close_approach_data"][0] if a["close_approach_data"] else {}
        registros.append({
            "fecha_acercamiento"         : fecha,
            "asteroide_id"               : a["id"],
            "nombre"                     : a["name"],
            "magnitud_absoluta"          : float(a.get("absolute_magnitude_h", 0)),
            "diametro_min_km"            : float(a["estimated_diameter"]["kilometers"]["estimated_diameter_min"]),
            "diametro_max_km"            : float(a["estimated_diameter"]["kilometers"]["estimated_diameter_max"]),
            "es_potencialmente_peligroso": a.get("is_potentially_hazardous_asteroid", False),
            "distancia_tierra_km"        : float(acercamiento.get("miss_distance", {}).get("kilometers", 0)),
            "velocidad_kmh"              : float(acercamiento.get("relative_velocity", {}).get("kilometers_per_hour", 0)),
            "cuerpo_orbitado"            : acercamiento.get("orbiting_body", "Earth"),
            "url_nasa"                   : a.get("nasa_jpl_url", ""),
            "ingestion_timestamp"        : datetime.now()
        })

df_bronze = spark.createDataFrame(registros)
print(f"📦 Registros listos para guardar: {df_bronze.count()}")
df_bronze.printSchema()

In [0]:
%python
# Guardamos los datos en Delta.
df_bronze.createOrReplaceTempView("neo_raw_temp")

spark.sql("""
    MERGE INTO nasa_dw.bronze.neo_raw AS tgt
    USING neo_raw_temp AS src
    ON tgt.asteroide_id = src.asteroide_id
    AND tgt.fecha_acercamiento = src.fecha_acercamiento
    WHEN NOT MATCHED THEN INSERT *
""")

print("✅ Datos guardados en nasa_dw.bronze.neo_raw sin duplicados")

In [0]:
%sql
-- Confirmamos que funcione.
SELECT
    fecha_acercamiento,
    nombre,
    ROUND(distancia_tierra_km, 0) AS distancia_km,
    ROUND(velocidad_kmh, 0)       AS velocidad_kmh,
    es_potencialmente_peligroso
FROM nasa_dw.bronze.neo_raw
ORDER BY fecha_acercamiento DESC
LIMIT 10;